In [1]:
import numpy as np
import pandas as pd
from math import factorial

df = pd.read_csv("full_wc_match_data.csv")
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 928 entries, 0 to 928
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   match_id    928 non-null    int64  
 1   year        928 non-null    int64  
 2   stage       928 non-null    object 
 3   home_team   928 non-null    object 
 4   away_team   928 non-null    object 
 5   home_score  928 non-null    float64
 6   away_score  928 non-null    float64
 7   home_rank   928 non-null    float64
 8   away_rank   928 non-null    float64
 9   result      928 non-null    int64  
 10  home_gdp    928 non-null    float64
 11  home_pop    928 non-null    float64
 12  away_gdp    928 non-null    float64
 13  away_pop    928 non-null    float64
dtypes: float64(8), int64(3), object(3)
memory usage: 108.8+ KB


In [2]:
def transform_features(row):
    rH = row["home_rank"]
    rA = row["away_rank"]

    gH = np.log(row["home_gdp"])
    gA = np.log(row["away_gdp"])

    pH = np.log(row["home_pop"])
    pA = np.log(row["away_pop"])

    xH = np.array([1, rH, gH, pH, rA, gA, pA, 1])
    xA = np.array([1, rA, gA, pA, rH, gH, pH, 0])

    return xH, xA


def train_poisson(df, epochs=500, lr=0.001):
    beta = np.zeros(8)

    for epoch in range(epochs):
        grad = np.zeros_like(beta)

        for _, row in df.iterrows():
            xH, xA = transform_features(row)

            lambdaH = np.exp(np.dot(beta, xH))
            lambdaA = np.exp(np.dot(beta, xA))

            kH = row["home_score"]
            kA = row["away_score"]

            grad += (kH - lambdaH) * xH
            grad += (kA - lambdaA) * xA

        beta += lr * grad

    return beta

def poisson_prob(k, lam):
    return (lam**k * np.exp(-lam)) / factorial(k)

def match_outcome_probs(lambdaH, lambdaA, max_goals=10):
    P_home, P_draw, P_away = 0, 0, 0

    for i in range(max_goals+1):
        for j in range(max_goals+1):
            p = poisson_prob(i, lambdaH) * poisson_prob(j, lambdaA)

            if i > j:
                P_home += p
            elif i == j:
                P_draw += p
            else:
                P_away += p

    return P_home, P_draw, P_away

def predict_lambdas(beta, row):
    xH, xA = transform_features(row)

    lambdaH = np.exp(np.dot(beta, xH))
    lambdaA = np.exp(np.dot(beta, xA))

    return lambdaH, lambdaA

def get_actual_result(row):
    if row["home_score"] > row["away_score"]:
        return 0  # home win
    elif row["home_score"] == row["away_score"]:
        return 1  # draw
    else:
        return 2  # away win


def compute_accuracy(df, beta):
    correct = 0

    for _, row in df.iterrows():
        lambdaH, lambdaA = predict_lambdas(beta, row)

        P_home, P_draw, P_away = match_outcome_probs(lambdaH, lambdaA)

        pred = np.argmax([P_home, P_draw, P_away])
        actual = get_actual_result(row)

        if pred == actual:
            correct += 1

    return correct / len(df)




In [3]:
# Train model
beta = train_poisson(df)

# Evaluate
accuracy = compute_accuracy(df, beta)

print("Accuracy:", accuracy)


C:\Users\akgun\AppData\Local\Temp\ipykernel_24404\1905062150.py:26: RuntimeWarning: overflow encountered in exp
  lambdaH = np.exp(np.dot(beta, xH))
C:\Users\akgun\AppData\Local\Temp\ipykernel_24404\1905062150.py:27: RuntimeWarning: overflow encountered in exp
  lambdaA = np.exp(np.dot(beta, xA))
C:\Users\akgun\AppData\Local\Temp\ipykernel_24404\1905062150.py:33: RuntimeWarning: invalid value encountered in multiply
  grad += (kA - lambdaA) * xA


Accuracy: 0.5463362068965517
